In [ ]:
import pandas as pd
country_codes = {
    "Albania" : "AL",
    "Austria" : "AT",
    "Bosnia and Herzegovina": "BA",
    "Belgium" : "BE",
    "Bulgaria" : "BG",
    "Switzerland" : "CH",
    "Cyprus" : "CY",
    "Czech Republic" : "CZ",
    "Germany" : "DE",
    "Denmark" : "DK",
    "Estonia" : "EE" ,
    "Spain" : "ES",
    "Finland" : "FI",
    "France" : "FR",
    "United Kingdom" : "UK",
    "Greece" : "GR",
    "Croatia" : "HR",
    "Hungary" : "HU",
    "Ireland" : "IE",
    "Iceland" : "IS",
    "Italy" : "IT",
    "Liechtenstein" : "LI",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Latvia" : "LV",
    "Montenegro" : "ME",
    "North Macedonia" : "MK",
    "Malta" : "MT",
    "Netherlands" : "NL",
    "Norway" : "NO",
    "Poland" : "PL",
    "Portugal" : "PT",
    "Romania" : "RO",
    "Serbia" : "RS",
    "Sweden" : "SE",
    "Slovenia" : "SI",
    "Slovakia" : "SK",
    "Kosovo" : "XK",
}
reversed_country_codes = {value: key for key, value in country_codes.items()}

In [ ]:
df = (
    pd.read_csv(snakemake.input.existing_capacities_full)
    .loc[:,["Technology","iso_code","capacity_MW","commissioning_year","estimated_retirement_year"]]
    .query("commissioning_year>0 and estimated_retirement_year>0")
)
df["country"] = df["iso_code"].replace(reversed_country_codes)



df_pairs = df[['country', 'Technology']].drop_duplicates().values
expected_multiindex = pd.MultiIndex.from_tuples(
    [tuple(pair) for pair in df_pairs],
    names=['country', 'Technology']
)
df

In [ ]:
start_year = 2025
end_year = int(snakemake.wildcards.year)

all_years = range(start_year, end_year + 1)

# existing plants until 2024 (included)
df_existing = (
    df
    .query("commissioning_year<=@start_year-1")
    .query("estimated_retirement_year>=@start_year-1")
    .loc[:,["Technology","country","capacity_MW"]]
    .rename(columns={"capacity_MW":2024})
    .groupby(["country","Technology"])
    .sum()
)
time_series_existing = df_existing.reindex(index=expected_multiindex, fill_value=0)


# all plants commissioning between 2025 and 2050 (included both)
df_commissioned_plan = (
    df
    .query("commissioning_year>=@start_year")
    .query("commissioning_year<=@end_year")
    .loc[:,["Technology","country","capacity_MW","commissioning_year"]]
    .groupby(["country","Technology","commissioning_year"])
    .sum()
)
# commissioned_capacity
time_series_commissioned = df_commissioned_plan["capacity_MW"].unstack(level='commissioning_year').fillna(0)
time_series_commissioned = time_series_commissioned.reindex(index=expected_multiindex,columns=all_years, fill_value=0)


# all plants retired between 2025 and 2050 (included both)
# df_retirement_plan = (
#     df
#     .query("estimated_retirement_year>=@start_year")
#     .query("estimated_retirement_year<=@end_year")
#     .loc[:,["Technology","country","capacity_MW","estimated_retirement_year"]]
#     .groupby(["country","Technology","estimated_retirement_year"])
#     .sum()
# )
df_retirement_plan = (
    df
    .query("estimated_retirement_year>=@start_year-1")
    .query("estimated_retirement_year<=@end_year-1")
    .loc[:,["Technology","country","capacity_MW","estimated_retirement_year"]]
    .groupby(["country","Technology","estimated_retirement_year"], as_index=False)
    .sum()
)
df_retirement_plan["estimated_retirement_year"] = df_retirement_plan["estimated_retirement_year"]+1
df_retirement_plan = (
    df_retirement_plan
    .set_index(["country","Technology","estimated_retirement_year"])
)

# decommissioned_capacity
time_series_decommissioned = df_retirement_plan["capacity_MW"].unstack(level='estimated_retirement_year').fillna(0)
time_series_decommissioned = time_series_decommissioned.reindex(index=expected_multiindex,columns=all_years, fill_value=0)


total_cap_year = time_series_commissioned-time_series_decommissioned
time_series_plan = time_series_existing.join(total_cap_year).cumsum(axis=1)
time_series_plan = time_series_plan.drop(columns=[2024])

time_series_plan


time_series_plan.to_csv(snakemake.output.time_series_path)